In [29]:
! pip install ultralytics

In [30]:
import os 
import random
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
import zipfile

In [31]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="iA4AfRz8sIrPum1ZNYjx")
project = rf.workspace("object-detection-od4vl").project("od-ke0lu")
version = project.version(9)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...


In [32]:
import shutil

src = "/kaggle/working/OD-7"
dst = "/kaggle/working/vehicle-detection-7_modifyed"

shutil.copytree(src, dst, dirs_exist_ok=True)



'/kaggle/working/vehicle-detection-7_modifyed'

In [33]:
from pathlib import Path

dataset_path = Path("/kaggle/working/OD-8")

for split in ["train", "valid", "test"]:

    labels_dir = dataset_path / split / "labels"

    if not labels_dir.exists():
        continue

    for label_file in labels_dir.glob("*.txt"):

        new_lines = []

        with open(label_file, "r") as f:

            for line in f:

                parts = line.strip().split()

                if not parts:
                    continue

                class_id = parts[0]

                coords = list(map(float, parts[1:]))

                if len(coords) < 6 or len(coords) % 2 != 0:
                    continue

                x_coords = coords[0::2]
                y_coords = coords[1::2]

                x_min = min(x_coords)
                x_max = max(x_coords)

                y_min = min(y_coords)
                y_max = max(y_coords)

                x_center = (x_min + x_max) / 2
                y_center = (y_min + y_max) / 2

                width = x_max - x_min
                height = y_max - y_min

                new_lines.append(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{width:.6f} "
                    f"{height:.6f}"
                )

        with open(label_file, "w") as f:
            f.write("\n".join(new_lines))

In [34]:
# label = list(
#     (dataset_path / "train" / "labels").glob("*.txt")
# )[0]

# print(label.read_text())

In [35]:
# labels_path = dataset_path / "train" / "labels"

# print(labels_path)
# print(labels_path.exists())

In [36]:
yamel_path = "/kaggle/working/OD-9/data.yaml"

In [37]:
model = YOLO("yolo26n.pt")

model.train(
    data=yamel_path,
    epochs=50,
    imgsz=640,
    patience=6
)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/OD-9/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, ops

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d9ec3454ce0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [38]:
best_model = YOLO("runs/detect/train/weights/best.pt")
metrics = best_model.val(data=yamel_path, split="test")

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1171.0±345.4 MB/s, size: 40.5 KB)
val: Scanning /kaggle/working/OD-9/test/labels... 434 images, 0 backgrounds, 10 corrupt: 100% ━━━━━━━━━━━━ 434/434 1.3Kit/s 0.3s<0.0s
val: /kaggle/working/OD-9/test/images/1554_jpg.rf.8247d11ef247c40042f90ad028b09e4a.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /kaggle/working/OD-9/test/images/1555_jpg.rf.c7893c0ef3010f794d995ed1ae17235b.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /kaggle/working/OD-9/test/images/1574_jpg.rf.7256318522c7a6783649b42878793872.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /kaggle/working/OD-9/test/images/1799_jpg.rf.3d4ceaec0548f1450ee5a3786dbdabbb.jpg: ignoring corrupt image/label: labels mix segment and detect

In [39]:
model.save("yolo.pt")